In [21]:
from pythreejs import *
import numpy as np
from IPython.display import display
from ipywidgets import HTML, Output, VBox, jslink
from ipywidgets.embed import embed_minimal_html

view_width = 800
view_height = 400

def f(x, y):
    distance_from_origin = 0.5*np.sqrt(x**2 + y**2)
    return 1-1 / (0.3 + distance_from_origin)

nx, ny = (51, 51)  # grid resolution
xmax = 5           # grid extent (+/-)
x = np.linspace(-xmax, xmax, nx)
y = np.linspace(-xmax, xmax, ny)
step = x[1] - x[0]
xx, yy = np.meshgrid(x, y)
# Grid lattice values:
grid_z = np.vectorize(f)(xx, yy)
# Grid square center values:
center_z = np.vectorize(f)(0.5 * step + xx[:-1,:-1], 0.5 * step + yy[:-1,:-1])

# Surface geometry:
surf_g = SurfaceGeometry(z=list(grid_z.flat), 
                         width=2 * xmax,
                         height=2 * xmax,
                         width_segments=nx - 1,
                         height_segments=ny - 1)

# Surface material. Note that the map uses the center-evaluated function-values:
surf = Mesh(geometry=surf_g,
            material=MeshLambertMaterial(map=height_texture(center_z, 'viridis')))

# Grid-lines for the surface:
surfgrid = SurfaceGrid(geometry=surf_g, 
                       material=LineBasicMaterial(color='gray'),
                       position=[0, 0, 1e-2])  # Avoid overlap by lifting grid slightly

# Set up scene:
key_light = DirectionalLight(color='white', position=[3, 5, 1], intensity=0.4)
c = PerspectiveCamera(position=[0, 3, 3], up=[0, 0, 1], aspect=view_width / view_height,
                      children=[key_light])
planet = Mesh(geometry=IcosahedronGeometry(radius=.25, detail=1, _flat=True), 
               material=LineBasicMaterial(color='#555555'),
               position=[0, 0, 0])

ls = LineSegments(
    EdgesGeometry(IcosahedronGeometry(radius=.25, detail=2, _flat=True)),
    LineBasicMaterial(parameters=dict(color='#555555')),
    position=[0, 0, 1]
)

#scene = Scene(children=[surf, c, surfgrid, ls, AmbientLight(intensity=0.8)], background='black')
scene = Scene(children=[c, surfgrid, ls, AmbientLight(intensity=0.8)], background='black')


def find_minima(f, start=(0, 0), xlim=None, ylim=None):
    rate = 0.1 # Learning rate
    max_iters = 200 # maximum number of iterations
    iters = 0 # iteration counter
    
    cur = np.array(start[:2])
    previous_step_size = 1 #
    cur_val = f(cur[0], cur[1]) 
    
    while (iters < max_iters and
           xlim[0] <= cur[0] <= xlim[1] and ylim[0] <= cur[1] <= ylim[1]):
        iters = iters + 1
        candidate = cur - rate * (np.random.rand(2) - 0.5)
        candidate_val = f(candidate[0], candidate[1])
        if candidate_val >= cur_val:
            continue   # Bad guess, try again
        prev = cur
        cur = candidate
        cur_val = candidate_val
        previous_step_size = np.abs(cur - prev)
        yield tuple(cur) + (cur_val,)

    print("The local minimum occurs at", cur)



renderer = Renderer(camera=c, scene=scene,
                    width=view_width, height=view_height,
                    controls=[OrbitControls(controlling=c)])

out = Output()        # An Output for displaying captured print statements
out.add_class("text_style")
box = VBox([renderer])
box.add_class("box_style")
embed_minimal_html('export.html', views=[box], title='Widget export')

style_html = HTML("""
<style>

.cell-output-ipywidget-background {
   background-color: transparent !important;
}
.jp-OutputArea-output {
   background-color: transparent;
}  

.box_style{
    background-color:transparent;
}
.text_style{
    color: white;
}
</style>
""")
display(style_html)
display(box)

# Picker object
hover_picker = Picker(controlling=surf, event='mousemove')
renderer.controls = renderer.controls + [hover_picker]

# A sphere for representing the current point on the surface
hover_point = Mesh(geometry=SphereGeometry(radius=0.05),
                   material=MeshLambertMaterial(color='hotpink'))
scene.add(hover_point)

# Have sphere follow picker point:
jslink((hover_point, 'position'), (hover_picker, 'point'));

coord_label = HTML()  # A label for showing hover picker coordinates

def on_hover_change(change):
    coord_label.value = '<p style="color:gray;">Pink point at (%.3f, %.3f, %.3f)</p>' % tuple(change['new'])

on_hover_change({'new': hover_point.position})
hover_picker.observe(on_hover_change, names=['point'])
box.children = (coord_label,) + box.children

# Create our picker for the double-click event ("dblclick")
click_picker = Picker(controlling=surf, event='dblclick')

def on_click(change):
    value = change['new']
    with out:
        print('Clicked on %s' % (value,))

    # Add a red sphere on the picked point
    point = Mesh(geometry=SphereGeometry(radius=0.05), 
                 material=MeshLambertMaterial(color='red'),
                 position=value)
    scene.add(point)
    
    # Plot solution as a red line, this will start out empty
    points = [value]
    line = Line2(geometry=LineGeometry(positions=points), material=LineMaterial(color='red', linewidth=2))
    scene.add(line)
    with out:  # Pick up any print statements in the algorithm
        for pt in find_minima(f, value, [-xmax, xmax], [-xmax, xmax]):
            # For each point, update the line:
            pt = list(pt)
            pt[2] += 1e-2   # offset to clear surface
            line.geometry = LineGeometry(positions=np.vstack([line.geometry.positions, pt]))


# When the point selected by the picker changes, trigger our function:
click_picker.observe(on_click, names=['point'])

# Update figure:
renderer.controls = renderer.controls + [click_picker]
box.children = box.children + (out,)
#export display as html file


HTML(value='\n<style>\n\n.cell-output-ipywidget-background {\n   background-color: transparent !important;\n}\…